# RemediSync: Data Understanding & Schema Integrity Audit
Automated audit and profiling of raw telemetry across 4 branches, 30 suppliers, and 129 SKUs.

In [10]:
import sys
from pathlib import Path
import json
import pandas as pd

# Resolve project root dynamically across all execution contexts
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not ((PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "src").exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

from src.features.data_audit import DataHealthAuditor

# Execute OOP health auditor
auditor = DataHealthAuditor(raw_dir=RAW_DIR, processed_dir=PROCESSED_DIR)
report = auditor.run_full_audit()
print(f"Audit Status: {report['audit_status']} | Critical Errors: {report['critical_errors_count']}")

Audit Status: PASSED | Critical Errors: 0


In [11]:
# Load raw tables for inspection
branches = pd.read_csv(RAW_DIR / "branches.csv")
suppliers = pd.read_csv(RAW_DIR / "suppliers.csv")
medicines = pd.read_csv(RAW_DIR / "medicines.csv")
batches = pd.read_csv(RAW_DIR / "batches.csv")
sales = pd.read_csv(RAW_DIR / "daily_sales.csv")
snapshot = pd.read_csv(RAW_DIR / "inventory_snapshot.csv")

# Summary table shapes and null counts
summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns), "null_count": int(df.isna().sum().sum())}
    for name, df in [
        ("branches", branches), ("suppliers", suppliers), ("medicines", medicines),
        ("batches", batches), ("daily_sales", sales), ("inventory_snapshot", snapshot)
    ]
])
summary

,table,rows,columns,null_count
0,branches,4,5,0
1,suppliers,30,6,0
2,medicines,129,12,0
3,batches,808,9,0
4,daily_sales,329208,7,0
5,inventory_snapshot,516,5,0


In [12]:
# Daily revenue profiling per branch (INR)
daily_branch_rev = sales.groupby(["date", "branch_id"])["total_revenue"].sum().reset_index()
daily_stats = daily_branch_rev.groupby("branch_id")["total_revenue"].agg(["mean", "std", "min", "max"]).round(2)
daily_stats.merge(branches[["branch_id", "branch_name", "location_type"]], on="branch_id")

,branch_id,mean,std,min,max,branch_name,location_type
0,BR001,56700.09,8143.04,37684.98,83615.42,Kamakhya Gate Branch,TRANSIT_HUB
1,BR002,56625.96,8287.70,38250.12,82608.95,Dharapur Branch,SUBURBAN_RESIDENTIAL
2,BR003,61466.76,9032.66,39769.24,90018.25,Azara Branch,HOSPITAL_PERIMETER
3,BR004,51759.30,7737.85,34759.04,76440.84,Rani Gate Branch,RESIDENTIAL_EDGE


In [13]:
# Batch expiry stratification (<90d, 90-365d, >365d)
batches["expiry_dt"] = pd.to_datetime(batches["expiry_date"])
sim_end = pd.to_datetime("2026-09-30")
batches["days_to_expiry"] = (batches["expiry_dt"] - sim_end).dt.days

batches["expiry_tier"] = pd.cut(
    batches["days_to_expiry"],
    bins=[-100, 90, 365, 1000],
    labels=["Near-Term (<90d)", "Medium-Term (90-365d)", "Long-Term (>365d)"]
)
batches["expiry_tier"].value_counts().reset_index()

,expiry_tier,count
0,Long-Term (>365d),560
1,Medium-Term (90-365d),191
2,Near-Term (<90d),57
